# Extended Kalman Filter, Fit with Multiple Time-Series

## Library and Configs

In [ ]:
import pandas as pd
import numpy as np

import numpyro
numpyro.set_host_device_count(4)
numpyro.enable_x64(True)
import jax.numpy as jnp

from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS

import matplotlib.pyplot as plt
from jax import random

from wunkui.models.ssp.kalman_1d_st import kalman_filter_1d_st
from wunkui.models.ssp.kalman_1d_st_ekf import kalman_filter_1d_ekf_st
from wunkui.models.ssp import transform_to_ekf


In [ ]:
df = pd.read_csv(
    '../resource/sparse/sparse_df.csv', parse_dates=['date']
)

media_vars = [
    "ch_1", "ch_2", "ch_3", "ch_4", "ch_5", "ch_6", "ch_7", "ch_8", "ch_9", "ch_10",
    "ch_11", "ch_12", "ch_13",
]

response = "outcome"
trend_vars = ["trend", "yearly", "weekly"]

time_point_beta_df = pd.read_csv("../resource/sparse/time_point_beta.csv")

restrict_positivity = True
num_warmup=50
num_samples=100
num_chains=4


In [ ]:
import xarray as xr

all_vars = media_vars + trend_vars + [response]

dates = sorted(df["date"].unique())
dmas  = list(df["dma"].unique())

ds = xr.Dataset(
    {
        v: xr.DataArray(
            df.pivot(index="date", columns="dma", values=v)
              .reindex(index=dates, columns=dmas)
              .values,
            coords={"date": dates, "series": dmas},
            dims=["date", "series"],
        )
        for v in all_vars
    }
)

## EDA

In [ ]:
def eda_plot(ds, response="outcome", trend_vars=("trend", "yearly", "weekly")):
    dates   = ds.coords["date"].values
    series  = ds.coords["series"].values

    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    for dma in series:
        y      = ds[response].sel(series=dma).values
        offset = sum(ds[v].sel(series=dma).values for v in trend_vars)
        resid  = y - offset
        norm_resid = resid / resid.std()

        axes[0].plot(dates, y,          label=dma, alpha=0.7)
        axes[1].plot(dates, offset,     label=dma, alpha=0.7)
        axes[2].plot(dates, norm_resid, label=dma, alpha=0.7)

    axes[0].set_title("response")
    axes[1].set_title("offset")
    axes[2].set_title("normalized residuals")
    axes[2].set_xlabel("date")
    for ax in axes:
        ax.legend(fontsize=7, ncol=4, loc="best")
    plt.tight_layout()
    return fig

_ = eda_plot(ds);

In [ ]:
# ── y: (date, series) — normalize per series ──────────────────────────────────
# (n_steps, n_series)
y_raw   = ds[response].values                        
# (n_steps, n_series, n_vars) -> sum(-1) -> (n_steps, n_series)
offset = ds[trend_vars].to_array().values.transpose([1, 2, 0]).sum(axis=-1)
# (n_series, ) — std per series                    
y_std = np.std(y_raw, axis=0)           
# (n_steps, n_series)                
# y = (y_raw - y_raw.mean(axis=0)) / y_std            
# y = (y_raw - offset) / y_std         
y = jnp.asarray(y_raw).astype(jnp.float64)

# original scale
Z = ds[media_vars].to_array().values.transpose([1, 2, 0])
# Z = (Z - Z.mean(axis=0)) / Z.std(axis=0) 
Z = np.concatenate([offset[:, :, None], Z], axis=-1)
Z = jnp.asarray(Z).astype(jnp.float64)

print("y shape:", y.shape)
print("Z shape:", Z.shape)

Definition of initial states and priors.

In [ ]:
n_steps, n_series, n_states = Z.shape
print(f"n_steps: {n_steps}, n_series: {n_series}, n_states: {n_states}")

In [ ]:
def derive_time_point_priors(
    time_point_beta_df: pd.DataFrame,
    media_vars: list[str],
    dates: np.ndarray,
    n_states: int,
    n_weeks: int = 4,
) -> tuple[jnp.ndarray, jnp.ndarray]:
    """Build a_obs / P_obs from time-point beta priors for media states.

    Every ``n_weeks`` weeks the media-state slots (indices 1..n_states-1)
    are filled with ``mid`` as location and ``3 * mid**2`` as variance.
    The offset state (index 0) and all non-anchor timesteps stay at
    loc=0, var=inf (zero precision → no-op in the filter).

    Parameters
    ----------
    time_point_beta_df : pd.DataFrame
        Columns: date, variable, mid, lower, upper.
    media_vars : list[str]
        Media variable names matching the ``variable`` column.
    dates : np.ndarray
        Sorted date array aligned with the observation matrix y.
    n_states : int
        Total number of latent states (offset + media).
    n_weeks : int, optional
        Interval in weeks between prior anchors. Default 4.

    Returns
    -------
    a_obs : jnp.ndarray, shape (n_steps, n_states)
    P_obs : jnp.ndarray, shape (n_steps, n_states)
    """
    n_steps = len(dates)
    dates_dt = pd.to_datetime(dates)

    a_obs = np.zeros((n_steps, n_states))
    P_obs = np.full((n_steps, n_states), np.inf)

    # pivot beta df → rows=date, cols ordered by media_vars, values=mid
    beta_pivot = (
        time_point_beta_df
        .loc[time_point_beta_df["variable"].isin(media_vars)]
        .pivot(index="date", columns="variable", values="mid")
        .reindex(columns=media_vars)
    )
    beta_pivot.index = pd.to_datetime(beta_pivot.index)

    # anchor every n_weeks (in days)
    step_size = n_weeks * 7
    for idx in range(0, n_steps, step_size):
        dt = dates_dt[idx]
        if dt not in beta_pivot.index:
            continue
        mid_vals = beta_pivot.loc[dt, media_vars].values.astype(float)
        # media states start at index 1 (index 0 is offset)
        a_obs[idx, 1:] = mid_vals
        # P_obs[idx, 1:] = (0.1 * mid_vals) ** 2
        P_obs[idx, 1:] = 0.01

    return jnp.asarray(a_obs), jnp.asarray(P_obs)


a_obs, P_obs = derive_time_point_priors(
    time_point_beta_df, media_vars, dates, n_states, n_weeks=4,
)
# offset unconstrained; media states must be non-negative
positivity_idx = jnp.array([True] + [True] * len(media_vars))

print("a_obs shape:", a_obs.shape)
print("P_obs shape:", P_obs.shape)
print("positivity_idx:", positivity_idx)
n_active = int(jnp.isfinite(P_obs[:, 1]).sum())
print(f"active prior anchors: {n_active} out of {a_obs.shape[0]} timesteps")

In [ ]:
first_dt_beta_media_vars = time_point_beta_df.loc[
 (time_point_beta_df["date"] == time_point_beta_df["date"].min())   &
    (time_point_beta_df["variable"].isin(media_vars))
]["mid"].values
assert len(first_dt_beta_media_vars) == len(media_vars), "Mismatch in number of media vars and priors"
first_dt_beta_media_vars = jnp.asarray(first_dt_beta_media_vars).astype(jnp.float64)
a0 = jnp.concatenate([jnp.asarray([1.0]), first_dt_beta_media_vars])
# a0 = jnp.zeros(n_states)

# create diagonal matrix with first element 1.0 and rest 0.1
P0 = jnp.diag(jnp.concatenate([jnp.asarray([1.0]), jnp.full(n_states - 1, 0.1)]))

In [ ]:
EXPONENT = 0.5

ssp_priors_nat = {
    "a0_nat": a0,
    "P0_nat": P0,
    # compressed (2,) = [offset, shared-media]
    "sigma_q_loc_prior_nat":   np.array([0.1, 0.01]),
    "sigma_q_scale_prior_nat": np.array([0.1, 0.01]),
    "sigma_q_low_prior_nat":   np.array([1e-5, 1e-5]),
    "sigma_q_high_prior_nat":  np.array([0.2,  0.1]),
    "a_obs_nat": np.array(a_obs),
    "P_obs_nat": np.array(P_obs),
    "obs_idx":   None,
}
ssp_priors_ekf = transform_to_ekf(ssp_priors_nat, positivity_idx, exponent=EXPONENT)

a0_ekf    = ssp_priors_ekf["a0"]
P0_ekf    = ssp_priors_ekf["P0"]
a_obs_ekf = ssp_priors_ekf["a_obs"]
P_obs_ekf = ssp_priors_ekf["P_obs"]

print("a0_ekf:", a0_ekf)
print("P0_ekf diag:", jnp.diag(P0_ekf))


## Model Fitting

Test run.

In [ ]:
def test_run(a0_ekf, P0_ekf, y, Z, positivity_idx):
    sigma_h = jnp.ones(n_series) * y_std.mean()
    sigma_q = jnp.ones(n_states) * 0.1

    lp, at, Pt, _, _, _ = kalman_filter_1d_ekf_st(
        a0=a0_ekf,
        P0=P0_ekf,
        sigma_h=sigma_h,
        sigma_q=sigma_q,
        y=y, Z=Z, logp=True,
        exponent=EXPONENT,
        positivity_idx=positivity_idx,
        a_obs=a_obs_ekf,
        P_obs=P_obs_ekf,
    )
    print("lp:", lp, "at shape:", at.shape)

test_run(a0_ekf, P0_ekf, y, Z, positivity_idx)


In [ ]:
def nuts_fn(a0_ekf, P0_ekf):
    with numpyro.plate("series", n_series):
        sigma_h_raw = numpyro.sample(
            "sigma_h_raw",
            dist.TruncatedNormal(0.1, 1.0, high=1.0, low=1e-3),
        )
    sigma_h = numpyro.deterministic("sigma_h", sigma_h_raw * y_std)

    # sample directly in a-space — priors already transformed by transform_to_ekf
    sigma_q_compressed = numpyro.sample(
        "sigma_q_compressed",
        dist.TruncatedNormal(
            ssp_priors_ekf["sigma_q_loc_prior"],
            ssp_priors_ekf["sigma_q_scale_prior"],
            low=ssp_priors_ekf["sigma_q_low_prior"],
            high=ssp_priors_ekf["sigma_q_high_prior"],
        ),
    )
    # expand compressed (2,) → (n_states,): [offset, media, media, ..., media]
    sigma_q = numpyro.deterministic(
        "sigma_q",
        jnp.concatenate([sigma_q_compressed[:1], jnp.full(n_states - 1, sigma_q_compressed[1])]),
    )

    lp, at, _, _, _, _ = kalman_filter_1d_ekf_st(
        a0=a0_ekf, P0=P0_ekf, Z=Z,
        sigma_h=sigma_h, sigma_q=sigma_q,
        y=y, logp=True,
        exponent=EXPONENT,
        positivity_idx=positivity_idx,
        a_obs=a_obs_ekf,
        P_obs=P_obs_ekf,
    )

    numpyro.factor("lp", lp)
    numpyro.deterministic("at", at)
    at_nat = jnp.where(positivity_idx[None, :], jnp.exp(EXPONENT * at), at)
    numpyro.deterministic("at_nat", at_nat)
    numpyro.deterministic("mu", jnp.sum(Z * at_nat[:, None, :], axis=-1))


## MCMC with NUTS

In [ ]:
nuts_kernel = NUTS(nuts_fn)
mcmc = MCMC(
    nuts_kernel,
    num_warmup=num_warmup,
    num_samples=num_samples,
    num_chains=num_chains,
)
rng_key = random.PRNGKey(0)
rng_keys = random.split(rng_key, 2)
rng_key = rng_keys[0]
mcmc_rng_key = rng_keys[1]
mcmc.run(mcmc_rng_key, a0_ekf, P0_ekf)


In [ ]:
# package up posteriors
samples = mcmc.get_samples(group_by_chain=True)

# state names match Z columns: [offset] + media_vars
state_names = ["offset"] + media_vars

posteriors = xr.Dataset(
    {
        "sigma_h": xr.DataArray(
            np.array(samples["sigma_h"]),
            dims=["chain", "draw", "series"],
            coords={"series": dmas},
        ),
        "sigma_q": xr.DataArray(
            np.array(samples["sigma_q"]),
            dims=["chain", "draw", "states"],
            coords={"states": state_names},
        ),
        "at": xr.DataArray(
            np.array(samples["at"]),
            dims=["chain", "draw", "date", "states"],
            coords={"date": dates, "states": state_names},
        ),
        # at_nat: natural-scale states (= at for linear filter; = exp(exponent * at) for EKF positivity states)
        "at_nat": xr.DataArray(
            np.array(samples["at_nat"]),
            dims=["chain", "draw", "date", "states"],
            coords={"date": dates, "states": state_names},
        ),
    }
)

posteriors


In [ ]:
from numpyro.diagnostics import summary

# print_summary covers R-hat and n_eff for all sites
mcmc.print_summary()
diag = summary(samples, prob=0.9)

# flag any R-hat > 1.01 or n_eff < 100
# use max(r_hat) / min(n_eff) for multi-dimensional params (at, lam, mu, etc.)
print("\n--- Convergence flags ---")
for param, stats in diag.items():
    rhat = float(np.asarray(stats["r_hat"]).max())
    neff = float(np.asarray(stats["n_eff"]).min())
    if rhat > 1.01 or neff < 100:
        print(f"  WARNING  {param:<20}  r_hat={rhat:.4f}  n_eff={neff:.1f}")

## Diagnostic

### States Plot

In [ ]:
# latent states at_nat: natural-scale posteriors → collapse chain/draw for quantiles
# (restrict_positivity=True:  positivity states are back-transformed via exp(exponent * at))
# (restrict_positivity=False: at_nat == at, already in natural/linear scale)
at_samples = posteriors["at_nat"].values  # (chain, draw, date, states)
n_ch, n_dr = at_samples.shape[:2]
at_flat = at_samples.reshape(n_ch * n_dr, len(dates), len(state_names))

at_lo  = np.quantile(at_flat, 0.05, axis=0)  # (date, states)
at_mid = np.quantile(at_flat, 0.50, axis=0)
at_hi  = np.quantile(at_flat, 0.95, axis=0)

n_cols = 4
n_rows = int(np.ceil(len(state_names) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3.2 * n_rows), sharex=True)
axes = axes.flatten()
plot_dates = np.array(dates)

for k, (ax, sname) in enumerate(zip(axes, state_names)):
    ax.plot(plot_dates, at_mid[:, k], color="darkgreen", linewidth=0.9, label="median")
    ax.fill_between(
        plot_dates, at_lo[:, k], at_hi[:, k],
        alpha=0.25, color="darkgreen", label="90% CI",
    )
    # overlay prior anchors on media states (index ≥ 1)
    # a_obs holds natural-scale mid values — consistent with at_nat
    if k >= 1:
        anchor_mask = np.isfinite(np.asarray(P_obs)[:, k])
        if anchor_mask.any():
            ax.scatter(
                plot_dates[anchor_mask],
                np.asarray(a_obs)[anchor_mask, k],
                s=14, color="crimson", marker="x", label="prior anchor", zorder=3,
            )
    ax.axhline(0.0, color="black", linewidth=0.5, linestyle=":")
    ax.set_title(sname, fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis="x", labelsize=7, rotation=30)

for ax in axes[len(state_names):]:
    ax.set_visible(False)

axes[0].legend(fontsize=7)
fig.suptitle("Latent states (natural scale) — posterior median + 90% CI", y=1.01)
plt.tight_layout()
plt.show()


### Attribution Plot

In [ ]:
# at_nat × Z contribution per state, summed over series
# using at_nat (natural-scale) so contributions are correctly interpreted regardless of restrict_positivity
Z_xr = xr.DataArray(
    np.array(Z),
    dims=["date", "series", "states"],
    coords={"date": dates, "series": dmas, "states": state_names},
)

# (chain, draw, date, states) * (date, series, states)
#   → (chain, draw, date, series, states) → sum series → (chain, draw, date, states)
contrib = (posteriors["at_nat"] * Z_xr).sum("series")

# flatten chain/draw for quantile computation
n_ch, n_dr = contrib.sizes["chain"], contrib.sizes["draw"]
contrib_flat = contrib.values.reshape(n_ch * n_dr, len(dates), len(state_names))  # (S, date, states)

# ── weekly aggregation ──────────────────────────────────────────────────────
# anchor to first date; each week is a 7-day interval
dates_pd = pd.to_datetime(dates)
week_idx = ((dates_pd - dates_pd[0]).days // 7).values  # (date,)
n_weeks  = int(week_idx.max()) + 1
week_dates = [dates_pd[0] + pd.Timedelta(days=7 * w) for w in range(n_weeks)]

# sum daily contributions within each week → (S, weeks, states)
contrib_weekly = np.stack(
    [contrib_flat[:, week_idx == w, :].sum(axis=1) for w in range(n_weeks)],
    axis=1,
)

lo  = np.quantile(contrib_weekly, 0.05, axis=0)   # (weeks, states)
mid = np.quantile(contrib_weekly, 0.50, axis=0)
hi  = np.quantile(contrib_weekly, 0.95, axis=0)

# one panel per state
n_cols = 4
n_rows = int(np.ceil(len(state_names) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3.2 * n_rows), sharex=True)
axes = axes.flatten()
plot_dates = np.array(week_dates)

for k, (ax, sname) in enumerate(zip(axes, state_names)):
    ax.plot(plot_dates, mid[:, k], color="steelblue", linewidth=0.9, label="median")
    ax.fill_between(plot_dates, lo[:, k], hi[:, k], alpha=0.25, color="steelblue", label="90% CI")
    ax.set_title(sname, fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis="x", labelsize=7, rotation=30)

for ax in axes[len(state_names):]:
    ax.set_visible(False)

axes[0].legend(fontsize=7)
fig.suptitle("at_nat × Zt summed over series (weekly) — posterior median + 90% CI", y=1.01)
plt.tight_layout()
plt.show()


### Model Fitness Plot

In [ ]:
posteriors_dict = mcmc.get_samples()

mu_samples       = np.array(posteriors_dict["mu"])            # (n_samples, n_steps, n_series)
sigma_h_samples  = np.array(posteriors_dict["sigma_h"])       # (n_samples, n_series)

# posterior predictive: add per-series obs noise in normalized space
eps_samples = np.random.normal(0, sigma_h_samples[:, None, :], size=mu_samples.shape)

# back-transform to original scale: undo  y = (y_raw - offset) / y_std
# mu_orig   = mu_samples * y_std[None, None, :] + offset[None, :, :]
mu_orig = mu_samples
# yhat_orig = (mu_samples + eps_samples) * y_std[None, None, :] + offset[None, :, :]
yhat_orig = mu_orig + eps_samples

mu_lo,   mu_mid,   mu_hi   = np.quantile(mu_orig,   q=[0.05, 0.50, 0.95], axis=0)  # (n_steps, n_series)
yhat_lo, yhat_mid, yhat_hi = np.quantile(yhat_orig, q=[0.05, 0.50, 0.95], axis=0)

plot_dates = np.array(ds.coords["date"].values)
n_cols = 4
n_rows = int(np.ceil(n_series / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3.2 * n_rows), sharex=True)
axes = axes.flatten()

for j, (ax, dma) in enumerate(zip(axes, dmas)):
    ax.scatter(plot_dates, y_raw[:, j], s=4, alpha=0.5, color="black", label="actual", zorder=3)
    ax.plot(plot_dates, offset[:, j], color="orange", linewidth=0.9, linestyle="--", label="offset")
    ax.plot(plot_dates, mu_mid[:, j], color="steelblue", linewidth=0.9, label="fitted (median)")
    ax.fill_between(
        plot_dates, yhat_lo[:, j], yhat_hi[:, j],
        alpha=0.20, color="steelblue", label="90% pred interval",
    )
    ax.set_title(dma, fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis="x", labelsize=7, rotation=30)

for ax in axes[n_series:]:
    ax.set_visible(False)

axes[0].legend(fontsize=7, loc="upper left")
fig.suptitle("Posterior fit vs actual — one panel per DMA", y=1.01)
plt.tight_layout()
plt.show()
